In [1]:
import numpy as np
import pandas as pd
import networkx as nx
import torch
import copy
import itertools

from pymatgen.core.structure import Structure
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.graphs import StructureGraph
from pymatgen.analysis import local_env

from networkx.algorithms.components import is_connected

from sklearn.metrics import accuracy_score, recall_score, precision_score

from torch_scatter import scatter

from p_tqdm import p_umap

import ast
#import the random function library
import random


In [2]:
def num_atoms_restriction(df, max_num_atoms):
    atomic_num_list = df['atomic_numbers'].apply(ast.literal_eval)
    num_atoms = [len(x) for x in atomic_num_list]
    len_mask = [x<=max_num_atoms for x in num_atoms]
    df = df[len_mask]
    return df

def source_restriction(df, source): 
    if source == "mp_20": 
        mp_20_df = df[df['material_id'].str.startswith('m')]
        return mp_20_df
    else:
        return df

In [3]:
input_file = "/home/gridsan/tmackey/cdvae/data/mp_20_dm/train.csv"

In [34]:
df = pd.read_csv(input_file)

/tmp/ipykernel_3598948/3708779858.py:1: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file)


In [35]:
train_fraction = 0.8

In [36]:
n = len(df)*train_fraction
#round n
n = round(n)

#sample n rows from the dataframe
df = df.sample(n=n)
print("using {} rows given a train_fraction of {}".format(n, train_fraction))

#impose a restriction on the number of atoms in the compounds being read in 
#note that this is seperate from the parameter written by Xie - that enforces bounds on the model
#this limits the data that's being used. 
max_num_atoms = 20
df = num_atoms_restriction(df, max_num_atoms)

print("using {} rows after imposing a restriction on the number of atoms".format(len(df)))

#impose a restriction on the source of the data
source = "mp_20"
df = source_restriction(df, source)

print("using {} rows after imposing a restriction on the source".format(len(df)))


using 43205 rows given a train_fraction of 0.8
using 32443 rows after imposing a restriction on the number of atoms
using 21691 rows after imposing a restriction on the source


In [37]:
#read in the graph pt file 
pt_file_path = input_file[:-4] + ".pt"
graph_dict = torch.load(pt_file_path)
graph_list = list(graph_dict.values())

In [38]:
niggli=True
primitive=False
graph_method='crystalnn'

In [39]:
df

,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,material_id,formation_energy_per_atom,band_gap,pretty_formula,e_above_hull,elements,...,xrd,xrd_peak_locations,xrd_peak_intensities,atomic_numbers,TsachID,disc_sim_xrd,lattice_system,lattice parameters,composition,lattice_parameters
1936,1936,1936.0,1936.0,34114.0,mp-1185608,0.031244,0.0000,LaGd3,0.031244,"['Gd', 'La']",...,DiffractionPattern\n$2\Theta$: [17.26070137 19...,"[17.260701368487535, 19.305891758554687, 24.48...","[0.08006007428467447, 0.25001161515617853, 0.0...","[57, 64, 64, 64]",LaGdGdGd6.29348586.29348586.2934858131.7870531...,[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,0.0,"[6.29348575, 6.29348575, 6.29348575]",NaN,0.0
18713,18713,18713.0,18713.0,20741.0,mp-12383,-1.479912,0.1091,Sc2Te3,0.000582,"['Sc', 'Te']",...,DiffractionPattern\n$2\Theta$: [13.51652985 14...,"[13.51652985309742, 14.265318844341838, 16.694...","[5.521932482290679, 1.060551321964653, 1.16014...","[21, 21, 21, 21, 21, 21, 21, 21, 52, 52, 52, 5...",ScScScScScScScScTeTeTeTeTeTeTeTeTeTeTeTe13.737...,[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,0.0,"[13.7370076, 13.0961927, 7.19923713]",NaN,0.0
16356,16356,16356.0,16356.0,26239.0,mp-1224155,-0.104189,0.0000,In3SnAu12,0.009800,"['Au', 'In', 'Sn']",...,DiffractionPattern\n$2\Theta$: [ 7.52815757 15...,"[7.528157571972511, 15.089053735533897, 16.537...","[0.023538592505689054, 0.002136489593841345, 1...","[49, 49, 49, 50, 79, 79, 79, 79, 79, 79, 79, 7...",InInInSnAuAuAuAuAuAuAuAuAuAuAuAu4.901785.36031...,[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,2.0,"[4.90178, 5.3603, 11.743179]",NaN,2.0
13355,13355,13355.0,13355.0,31981.0,mp-982559,-0.717130,0.0000,HoAlAu2,0.023949,"['Al', 'Au', 'Ho']",...,DiffractionPattern\n$2\Theta$: [22.60072513 26...,"[22.60072512832476, 26.154702713716652, 37.324...","[12.635333712409428, 15.164761242516956, 100.0...","[67, 13, 79, 79]",HoAlAuAu4.81841794.81841794.818417960.060.060.0,[ 0. 0. 0. 0. ...,4.0,"[4.81841794, 4.81841794, 4.81841794]",NaN,4.0
1690,1690,1690.0,1690.0,684.0,mp-9010,-0.927913,2.0942,RbAuS,0.000000,"['Rb', 'Au', 'S']",...,DiffractionPattern\n$2\Theta$: [17.05605313 21...,"[17.056053132331755, 21.183930044654186, 21.77...","[72.00416763311308, 3.3486960655443867, 27.946...","[37, 37, 79, 79, 16, 16]",RbRbAuAuSS5.29380675.29380677.09793590.090.010...,[ 0. 0. 0. 0. ...,0.0,"[5.29380672, 5.29380672, 7.097935]",NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13186,13186,13186.0,13186.0,8857.0,mp-36262,-0.639713,2.8995,BrO2F,0.000000,"['Br', 'F', 'O']",...,DiffractionPattern\n$2\Theta$: [19.71988642 21...,"[19.719886415333214, 21.495160264612032, 21.58...","[0.6023799208835966, 100.0, 51.451059159601655...","[35, 35, 8, 8, 8, 8, 9, 9]",BrBrOOOOFF5.40075755.40075756.699002965.573281...,[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,0.0,"[5.40075751, 5.40075751, 6.69900289]",NaN,0.0
8542,8542,8542.0,8542.0,14751.0,mp-19215,-2.432014,2.3714,CaCrO4,0.000000,"['Ca', 'Cr', 'O']",...,DiffractionPattern\n$2\Theta$: [18.50151106 24...,"[18.501511059870932, 24.22222886149529, 30.652...","[7.1776062806832766, 100.0, 13.385610441018333...","[20, 20, 24, 24, 8, 8, 8, 8, 8, 8, 8, 8]",CaCaCrCrOOOOOOOO6.08409926.08409926.0840992105...,[0.00000000e+00 0.00000000e+00 0.00000000e+00 ...,0.0,"[6.08409919, 6.08409919, 6.08409919]",NaN,0.0
14960,14960,14960.0,14960.0,17918.0,mp-7956,-0.476247,0.3945,Na3Sb,0.000000,"['Na', 'Sb']",...,DiffractionPattern\n$2\Theta$: [18.56140034 19...,"[18.56140033921242, 19.0626661199378, 21.22630...","[47.890594515813184, 33.96881152620915, 100.0,...","[11, 11, 11, 11, 11, 11, 51, 51]",NaNaNaNaNaNaSbSb5.3759175.3759179.5605390.090....,[ 0. 0. 0. 0. ...,5.0,"[5.37591702, 5.37591702, 9.56053]",NaN,5.0
21418,21418,21418.0,21418.0,2808.0,mp-30991,-1.546154,3.5469,Ba(IO3)2,0.000000,"['Ba', 'I', 'O']",...,DiffractionPattern\n$2\Theta$: [13.82871659 14...,"[13.82871658930776, 14.848826161462446, 17.857...","[5.658022911687461, 71.766609

In [40]:
prop_list = ['band_gap']

In [41]:
num_workers = 1
#do xrd_peak_intensities first 
#apply ast.literal_eval to the xrd_peak_intensities column
df['xrd_peak_intensities'] = df['xrd_peak_intensities'].apply(ast.literal_eval)
df['xrd_peak_intensities'] = df['xrd_peak_intensities'].apply(lambda x: (x[:256] + [0]*256)[:256])
xrd_intensities = torch.stack([torch.tensor(x) for x in df['xrd_peak_intensities']])
#repeat for xrd_peak_locations
df['xrd_peak_locations'] = df['xrd_peak_locations'].apply(ast.literal_eval)

df['xrd_peak_locations'] = df['xrd_peak_locations'].apply(lambda x: (x[:256] + [0]*256)[:256])
xrd_locations = torch.stack([torch.tensor(x) for x in df['xrd_peak_locations']])
#for the atomic numbers, we first need to do as lit eval
df['atomic_numbers'] = df['atomic_numbers'].apply(ast.literal_eval)
df['atomic_numbers'] = df['atomic_numbers'].apply(lambda x: list(set(x)))
#randomly shuffle the atomic numbers
df['atomic_numbers'] = df['atomic_numbers'].apply(lambda x: random.sample(x, len(x)))
#then, we need to pad the list with zeros
df['atomic_numbers'] = df['atomic_numbers'].apply(lambda x: (x + [0]*256)[:256])
#then, we need to convert to a tensor
atomic_numbers = torch.stack([torch.tensor(x) for x in df['atomic_numbers']])
#repeat the process for disc sim xrd 
df['disc_sim_xrd'] = df['disc_sim_xrd'].apply(lambda x: [float(val) for val in x[1:-1].split() if val])

#pad and stack
df['disc_sim_xrd'] = df['disc_sim_xrd'].apply(lambda x: (x + [0]*256)[:256])
disc_sim_xrd = torch.stack([torch.tensor(x) for x in df['disc_sim_xrd']])

#get the properties
prop_list = ['band_gap']

prop_dictionary = {}
for prop in prop_list:
    prop_dictionary[prop] = df[prop].values.astype(np.float32)
#merge everything into a list of dictionaries
ordered_results = []
for i in range(len(df)):
    materials_id = df['material_id'].iloc[i]
    ordered_results.append({'xrd_intensities': xrd_intensities[i], 'xrd_locations': xrd_locations[i], 
                      'atomic_numbers': atomic_numbers[i], 'disc_sim_xrd': disc_sim_xrd[i], 'graph_arrays': graph_dict[materials_id]})
    for prop in prop_list:
        ordered_results[i][prop] = prop_dictionary[prop][i]

In [42]:
ordered_results[0]

{'xrd_intensities': tensor([8.0060e-02, 2.5001e-01, 7.2739e-02, 1.0000e+02, 8.0358e-02, 3.3821e+01,
         1.6878e+01, 4.5798e-02, 8.5723e-02, 3.5564e-02, 5.4238e-02, 2.7089e-02,
         1.2107e+01, 2.4187e+01, 1.9620e-02, 4.8960e-03, 1.8815e-02, 1.6675e-02,
         2.8814e+01, 1.4383e+01, 1.3850e-02, 2.7679e-02, 1.2441e+01, 1.0998e-02,
         2.1373e-02, 1.0666e-02, 1.9582e-02, 1.7147e-02, 8.5686e-03, 3.8449e+00,
         1.9182e+00, 7.2868e-03, 7.2800e-03, 1.4271e-02, 1.4252e-02, 3.3601e-03,
         6.7089e-03, 5.8971e+00, 1.1785e+01, 1.2288e-02, 1.2284e-02, 5.4925e+00,
         5.4904e+00, 5.4845e+00, 1.1082e-02, 5.4669e-03, 5.4639e-03, 5.2810e-03,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         

In [25]:
def process_one(row, niggli, primitive, graph_method, prop_list, graph_list):
    crystal_str = row['cif']

    extra_feature_names = ['xrd_peak_intensities', 'xrd_peak_locations', 'atomic_numbers', 'disc_sim_xrd']
    extra_feature_data = []

    for feature in extra_feature_names:
        
        #some extra processing for disc_sim_xrd
        if feature == 'disc_sim_xrd':
            s = row[feature][1:-1]

            # Split the string based on spaces
            row[feature] = [float(val) for val in s.split() if val]
            feature_val_rough = row[feature]
        else:

            #use ast to convert string to list
            feature_val_rough = ast.literal_eval(row[feature])

        #if the feature is atomic_numbers, get only the unique values and randomize
        if feature == 'atomic_numbers':
            # get the unique values
            feature_val_rough = list(set(feature_val_rough))
            # randomize the order 
            random.shuffle(feature_val_rough)

        #ensure that the vector is 256 long with 0 padding
        if len(feature_val_rough) < 256:
            feature_val_refined = feature_val_rough + [0]*(256-len(feature_val_rough))
        else:
            feature_val_refined = feature_val_rough[:256]

        #create tensors for everything 
        feature_tensor = torch.tensor(feature_val_refined)
        extra_feature_data.append(feature_tensor)
    # print(extra_feature_data)
    # crystal = build_crystal(
    #     crystal_str, niggli=niggli, primitive=primitive)
    # graph_arrays = build_crystal_graph(crystal, graph_method)
    # graph_arrays = graph_dict[(row['material_id'])]
    properties = {k: row[k] for k in prop_list if k in row.keys()}
    result_dict = {
        'mp_id': row['material_id'],
        'cif': crystal_str,
        # 'graph_arrays': graph_arrays,
        'xrd_intensities': extra_feature_data[0],
        'xrd_locations': extra_feature_data[1],
        'atomic_species': extra_feature_data[2],
        'disc_sim_xrd': extra_feature_data[3],
    }
    # print(result_dict)
    result_dict.update(properties)
    return result_dict
# print(df)

input_rows = [df.iloc[idx] for idx in range(len(df))]

unordered_results = p_umap(
    process_one,
    input_rows,
    [niggli] * len(df),
    [primitive] * len(df),
    [graph_method] * len(df),
    [prop_list] * len(df),
    graph_list,
    num_cpus=num_workers,)

mpid_to_results = {result['mp_id']: result for result in unordered_results}
ordered_results = [mpid_to_results[df.iloc[idx]['material_id']]
                    for idx in range(len(df))]

#add the graph data to ordered_reults
#note that this works because graph data is in the same order as the input_rows
for idx, result in enumerate(ordered_results):
    result["graph_arrays"] = graph_list[idx]

  0%|          | 0/17476 [00:00<?, ?it/s]

KeyboardInterrupt: 